In [20]:
import cv2
import numpy as np
import os
import joblib
import mediapipe as mp
import tensorflow as tf
import keras # For compatibility hack
from scipy.spatial.distance import cosine # For distance metric
from deepface import DeepFace 

In [21]:
# --- CRITICAL CONFIGURATION ---
MODEL_DIR = '../saved_models'
# Use an image that is NOT in your training data to test GENERALIZATION
TEST_IMAGE_PATH = '../test_images/IMG_8419.JPG' 
ALIGNMENT_SIZE = (224, 224) 
SIMILARITY_THRESHOLD = 0.40 # Max acceptable Cosine Distance (TUNE THIS: try 0.35 - 0.50)

# Inject compatibility layer for older DeepFace versions (TF 2.15)
if not hasattr(tf, 'keras'):
    tf.keras = keras


In [22]:
# --- 1. Load Deployment Assets (Averaged Templates) ---
try:
    # Load the averaged feature templates and labels
    X_AVG = joblib.load(os.path.join(MODEL_DIR, 'X_avg.pkl'))
    LABELS_MAP = joblib.load(os.path.join(MODEL_DIR, 'labels_to_names.pkl'))
    
    # Create map for ID -> Name lookup
    ID_TO_NAME = {i: name for i, name in enumerate(LABELS_MAP.values())} 

except FileNotFoundError:
    print("FATAL: Cannot load required PKL files. Check paths and ensure 02_train_model was run.")
    exit()

# Pre-load the DeepFace VGG-Face model for feature extraction
DeepFace.build_model('VGG-Face') 
mp_face_detection = mp.solutions.face_detection

print(f"DEBUG: System ready. Loaded {len(X_AVG)} templates.")

DEBUG: System ready. Loaded 2 templates.


In [23]:
# --- 2. Feature Extraction Pipeline (Helper) ---

def preprocess_and_extract_features(image_path):
    """Runs Detection, Alignment, and DeepFace Embedding."""
    
    img = cv2.imread(image_path)
    if img is None:
        print(f"DEBUG CRITICAL: cv2.imread failed for {image_path}.")
        return None, None, None
    
    with mp_face_detection.FaceDetection(
        model_selection=1, min_detection_confidence=0.7) as face_detection:
        
        results = face_detection.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        
        if not results.detections:
            print("Detection Failed: No face found by MediaPipe.")
            return None, None, None
        
        # Get coordinates, clamp, resize (using logic from 01_data_prep)
        detection = results.detections[0]
        bbox_c = detection.location_data.relative_bounding_box
        h, w, _ = img.shape
        
        xmin = max(0, int(bbox_c.xmin * w))
        ymin = max(0, int(bbox_c.ymin * h))
        xmax = min(w, int(bbox_c.width * w) + xmin)
        ymax = min(h, int(bbox_c.height * h) + ymin)

        face_img = img[ymin:ymax, xmin:xmax]
        face_aligned_resized = cv2.resize(face_img, ALIGNMENT_SIZE) 

        # DeepFace Embedding
        try:
            embedding_result = DeepFace.represent(
                img_path=face_aligned_resized, 
                model_name='VGG-Face', 
                enforce_detection=False, 
                detector_backend='skip'  
            )
            input_vector = np.array(embedding_result[0]['embedding']).flatten()
            return input_vector, (xmin, ymin, xmax, ymax), img 
        except Exception as e:
            print(f"Feature Extraction Failed (DeepFace Error): {e}")
            return None, None, None


In [24]:
# --- 3. Core Recognition Logic (Verification with Threshold) ---

def recognize_and_verify(input_vector):
    """Compares the input vector against templates and applies the threshold."""
    
    best_match_distance = float('inf')
    best_match_index = -1
    
    # 1. Calculate Distance to All Templates
    for i, avg_template in enumerate(X_AVG):
        # Cosine distance: 0.0 is perfect match
        distance = cosine(input_vector, avg_template)
        
        if distance < best_match_distance:
            best_match_distance = distance
            best_match_index = i
            
    # 2. Verification Check (Threshold Logic)
    
    if best_match_distance < SIMILARITY_THRESHOLD:
        # Verified! Look up the name
        predicted_name = ID_TO_NAME.get(best_match_index, "Unknown (DB Error)")
        
        print(f"\nMatch Found! Predicted: {predicted_name}")
        print(f"Best Distance (Cosine): {best_match_distance:.4f} (Threshold: <{SIMILARITY_THRESHOLD:.4f})")
        return predicted_name
    
    else:
        # Rejected! Distance is too high (Likely an imposter or face highly obscured)
        print(f"\nVerification Failed. Closest Distance: {best_match_distance:.4f} (Threshold exceeded)")
        return "Unknown"

In [25]:
# --- 4. EXECUTION ---

if __name__ == '__main__':
    
    # Ensure you set TEST_IMAGE_PATH to a valid image file!
    print(f"Testing image: {os.path.abspath(TEST_IMAGE_PATH)}")
    
    input_vector, bbox, original_img = preprocess_and_extract_features(TEST_IMAGE_PATH)
    
    if input_vector is not None:
        result = recognize_and_verify(input_vector)
        print(f"\nFINAL PREDICTION RESULT: {result}")
        
        # --- OPTIONAL: Visual Debug Display ---
        if bbox is not None and original_img is not None:
            xmin, ymin, xmax, ymax = bbox
            cv2.rectangle(original_img, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)
            cv2.putText(original_img, result, (xmin, ymin - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
            
            # Display image (Only run this part if you have a display environment)
            print("Displaying image. Press any key to close window.")
            cv2.imshow('Recognition Test', original_img)
            cv2.waitKey(0)
            cv2.destroyAllWindows()
    else:
        print("\nFINAL PREDICTION RESULT: System failed before prediction.")

Testing image: d:\Kao Vichet\Term 1 Year 4\Foundation of Machine Learning\Machine Learning Project\ml_research\test_images\IMG_8419.JPG

Verification Failed. Closest Distance: 0.4172 (Threshold exceeded)

FINAL PREDICTION RESULT: Unknown
Displaying image. Press any key to close window.
